Checkpoint 2: Data preprocessing + Basic data exploration and summary statistics

Data Preprocessing for the raw data from the Storm Events Database:

In [1]:
import pandas as pd
from datetime import datetime
import string

# Create the original database from the csv of the first year we're using. (1991)
storm_events_df = pd.read_csv('Storm_Events_CSVs/StormEvents_details-ftp_v1.0_d1950_c20250401.csv')

# Go through the folder containing the rest of the CSVs and add their data onto the complete database one by one.
for i in range(1951, 2020):
    year_storms_csv = f"Storm_Events_CSVs/StormEvents_details-ftp_v1.0_d{str(i)}_c20250520.csv"
    year_storms_df = pd.read_csv(year_storms_csv)
    storm_events_df = pd.concat([storm_events_df, year_storms_df], ignore_index=True)

# 2020 is different because the csv file was made at a different time so it has a different naming convention.
final_year = pd.read_csv('Storm_Events_CSVs/StormEvents_details-ftp_v1.0_d2020_c20240620.csv')
storm_events_df = pd.concat([storm_events_df, final_year], ignore_index=True)

# Function uses the BEGIN_YEARMONTH and BEGIN_DAY columns to create a datetime for when the storm took place.
def find_datetime(column1, column2):
    final_column = []
    for i in range(0, len(storm_events_df)):
        year_month = str(storm_events_df.loc[i, column1])
        year = year_month[0:4]
        month = year_month[4:]
        day = str(storm_events_df.loc[i, column2])
        if len(day) == 1:
            day = "0" + day
        new_string = year + "-" + month + "-" + day
        final_datetime = pd.to_datetime(new_string, format = '%Y-%m-%d', errors ='coerce')
        final_column.append(final_datetime)
    return pd.Series(final_column)

# Use the function above to create the EVENT_DATE column, and create columns for the total deaths and total injuries caused.
storm_events_df['EVENT_DATE'] = find_datetime('BEGIN_YEARMONTH','BEGIN_DAY')
storm_events_df['DEATHS_TOTAL'] = storm_events_df['DEATHS_DIRECT'] + storm_events_df['DEATHS_INDIRECT']
storm_events_df['INJURIES_TOTAL'] = storm_events_df['INJURIES_DIRECT'] + storm_events_df['INJURIES_INDIRECT']

# Create the final database by filtering out a lot of the unnecessary (for our purposes) columns.
storm_events_df = storm_events_df[['YEAR', 'EVENT_ID', 'EVENT_TYPE', 'EVENT_DATE', 'INJURIES_DIRECT', 'INJURIES_INDIRECT', 'INJURIES_TOTAL',
                  'DEATHS_DIRECT', 'DEATHS_INDIRECT', 'DEATHS_TOTAL', 'DAMAGE_PROPERTY', 'DAMAGE_CROPS']]

storm_events_df

/var/folders/sz/pzpm4f017q9bs9hbsd6y4g8c0000gn/T/ipykernel_90055/3496292672.py:11: DtypeWarning: Columns (26,48) have mixed types. Specify dtype option on import or set low_memory=False.
  year_storms_df = pd.read_csv(year_storms_csv)
/var/folders/sz/pzpm4f017q9bs9hbsd6y4g8c0000gn/T/ipykernel_90055/3496292672.py:11: DtypeWarning: Columns (26,28) have mixed types. Specify dtype option on import or set low_memory=False.
  year_storms_df = pd.read_csv(year_storms_csv)
/var/folders/sz/pzpm4f017q9bs9hbsd6y4g8c0000gn/T/ipykernel_90055/3496292672.py:11: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  year_storms_df = pd.read_csv(year_storms_csv)
/var/folders/sz/pzpm4f017q9bs9hbsd6y4g8c0000gn/T/ipykernel_90055/3496292672.py:11: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  year_storms_df = pd.read_csv(year_storms_csv)
/var/folders/sz/pzpm4f017q9bs9hbsd6y4g8c0000gn/T/ipykernel_90055/3

,YEAR,EVENT_ID,EVENT_TYPE,EVENT_DATE,INJURIES_DIRECT,INJURIES_INDIRECT,INJURIES_TOTAL,DEATHS_DIRECT,DEATHS_INDIRECT,DEATHS_TOTAL,DAMAGE_PROPERTY,DAMAGE_CROPS
0,1950,10096222,Tornado,1950-04-28,0,0,0,0,0,0,250K,0
1,1950,10120412,Tornado,1950-04-29,0,0,0,0,0,0,25K,0
2,1950,10104927,Tornado,1950-07-05,2,0,2,0,0,0,25K,0
3,1950,10104928,Tornado,1950-07-05,0,0,0,0,0,0,2.5K,0
4,1950,10104929,Tornado,1950-07-24,0,0,0,0,0,0,2.5K,0
...,...,...,...,...,...,...,...,...,...,...,...,...
1663919,2020,919277,Thunderstorm Wind,2020-08-10,0,0,0,0,0,0,NaN,NaN
1663920,2020,904958,Tornado,2020-06-02,0,0,0,0,0,0,0.00K,0.00K
1663921,2020,904282,Thunderstorm Wind,2020-06-08,0,0,0,0,0,0,NaN,0.00K
1663922,2020,896642,Hail,2020-06-02,0,0,0,0,0,0,NaN,0.00K


In [2]:
print(storm_events_df['YEAR'].count())
print(storm_events_df['EVENT_ID'].count())
print(storm_events_df['EVENT_TYPE'].count())
print(storm_events_df['EVENT_DATE'].count())
print(storm_events_df['INJURIES_DIRECT'].count())
print(storm_events_df['INJURIES_INDIRECT'].count())
print(storm_events_df['INJURIES_TOTAL'].count())
print(storm_events_df['DEATHS_DIRECT'].count())
print(storm_events_df['DEATHS_INDIRECT'].count())
print(storm_events_df['DEATHS_TOTAL'].count())
print(storm_events_df['DAMAGE_PROPERTY'].count())
print(storm_events_df['DAMAGE_CROPS'].count())

1663924
1663924
1663924
1663924
1663924
1663924
1663924
1663924
1663924
1663924
1129298
1018239


In [3]:
# Function for converting the damage costs into numerical values and removing inconsistencies.
def standardize_costs(value):
    if type(value) is str:
        units = value[len(value) - 1]
        if units == 'h' or units == 'H':
            number = float(value[0:len(value) - 1])
            return (100 * number)
        elif units == 'k' or units == 'K':
            if len(value) == 1:
                number = 0
            else:
                number = float(value[0:len(value) - 1])
            return (1000 * number)
        elif units == 'm' or units == 'M':
            if len(value) == 1:
                number = 0
            else:
                number = float(value[0:len(value) - 1])
            return (1000000 * number)
        elif units == 'b' or units == 'B':
            number = float(value[0:len(value) - 1])
            return (1000000000 * number)
        elif units == 't' or units == 'T':
            number = float(value[0:len(value) - 1])
            return (1000000000000 * number)
        elif units == '?':
            number = float(value[0:len(value) - 1])
            return number
        else:
            number = float(value)
            return number
    else:
        return value

# Replace NaN values with 0 in the columns.
storm_events_df['DAMAGE_PROPERTY'].fillna(0)
storm_events_df['DAMAGE_CROPS'].fillna(0)

storm_events_df['DAMAGE_PROPERTY'] = storm_events_df['DAMAGE_PROPERTY'].apply(standardize_costs)
storm_events_df['DAMAGE_CROPS'] = storm_events_df['DAMAGE_CROPS'].apply(standardize_costs)

For the data prepocessing, the storm events and their details were originally separated by year in different csv files for each year. We put all of the data (from the US storm events database provided by the NOAA: https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/)
into one DataFrame by downloading them and adding them to the final database one by one. Then the find_datetime() function is used to create an actual datetime to track when the storm began because then we can remove a lot of the extra columns that have to do with the date that are useless and moastly repetitive. Then we created columns for the total deaths and injuries by adding together the direct and indirect columns for each. The columns for property and crop damage also had a lot of missing or inconsistent data, so we replaced the NaN values with 0, and made a function to make the data consistent. Finally, we filtered out the data that we didn't need, and only really kept the data describing how much damage the storm did.

(Can edit) Also, we are no longer using the 30-year average annual U.S. climate normals as a part of our dataset since those don't fit with the database when trying to look at each storm year-by-year.


Basic data exploration and summary statistics: